# 17 - Multitask EF Model Temporal Grad-CAM Evaluation

This notebook evaluates temporal behavior of EF Grad-CAM explanations for the two notebook-12 bidirectional ConvLSTM U-Net models:

- `ef_primary`: EF regression + LV segmentation
- `ef_primary_motion`: EF regression + LV segmentation + auxiliary motion head

It reuses the EF Grad-CAM implementation from notebook 14 and computes temporal saliency metrics on the official EchoNet-Dynamic test set.


## Setup

Keep `RUN_MODE = "smoke"` for a quick Kaggle validation run. Switch to `"full"` only after smoke mode completes. The notebook saves full Grad-CAM NPZ files by default because these are needed for reproducibility; if disk becomes a problem, set `CONFIG["save_gradcam_npz"] = False` and keep the CSV metrics.


In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import math
import os
import sys
from typing import Any

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, Subset
from tqdm.auto import tqdm

try:
    from scipy import stats as scipy_stats
except Exception:
    scipy_stats = None


def first_existing_path(candidates):
    cleaned = [candidate for candidate in candidates if candidate]
    for candidate in cleaned:
        path = Path(candidate)
        if path.exists():
            return path
    return Path(cleaned[-1])


PROJECT_ROOT = first_existing_path([
    os.environ.get("PROJECT_ROOT"),
    "/kaggle/input/echonet-temporal-xai",
    "/kaggle/input/src-updated",
    "/kaggle/input/datasets/jiyoonoh24/echonet-src-code",
    "/kaggle/input/datasets/sooahnoh/echonet-code-updated",
    "/kaggle/working/Echonet_temporal_XAI",
    Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd(),
])
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.dataset import EchoNetTemporalDataset, load_temporal_metadata, split_by_echonet_filelist
from src.bidirectional_convlstm_unet import build_bidirectional_convlstm_unet
from src.gradcam_ef_regression import (
    EchoNetTemporalEFDataset,
    build_ef_regression_convlstm,
    encoder_bottleneck_ef_gradcam,
    load_exact_checkpoint,
    make_centroid_trajectory_plot,
    make_gradcam_overlay_figure,
    make_motion_trace_overlay_figure,
    make_temporal_diagnostic_plot,
    regression_metrics,
    run_normal_inference,
    save_gradcam_npz,
    select_representative_samples,
    temporal_representation_ef_probe_gradcam,
)
from src.temporal_evaluation import adjacent_pearson, centroid_from_map
from src.utils import load_echonet_tables, set_seed

RAW_DIR = Path(os.environ.get("ECHONET_RAW_DIR", PROJECT_ROOT / "data" / "raw" / "EchoNet-Dynamic"))
PROCESSED_DIR = Path(os.environ.get("ECHONET_PROCESSED_DIR", PROJECT_ROOT / "data" / "processed"))
VIDEOS_DIR = RAW_DIR / "Videos"
TRAINED_RUN_DIR = Path(os.environ.get(
    "EF_MOTION_CONVLSTM_RUN_DIR",
    "/kaggle/input/ef-primary-motion-head-conv-lstm-07-22" if Path("/kaggle/input").exists() else PROJECT_ROOT / "outputs" / "runs" / "ef_primary_motion_head_conv_lstm_07_22",
))
SEGMENTATION_RUN_DIR = Path(os.environ.get(
    "BIDIRECTIONAL_CONVLSTM_SEG_RUN_DIR",
    "/kaggle/input/bidirectional-convlstm-unet-23-frames" if Path("/kaggle/input").exists() else PROJECT_ROOT / "outputs" / "runs" / "bidirectional_convlstm_unet_23_frames",
))
REFERENCE_GRADCAM_RUN_DIR = Path(os.environ.get(
    "REFERENCE_EF_GRADCAM_RUN_DIR",
    "/kaggle/input/ef-gradcam-convlstm-multitask-run-2-07-22" if Path("/kaggle/input").exists() else PROJECT_ROOT / "outputs" / "runs" / "ef_gradcam_convlstm_multitask_run_2_07_22",
))
RUN_DIR = Path(os.environ.get(
    "TEMPORAL_EF_GRADCAM_EVAL_RUN_DIR",
    "/kaggle/working/outputs/runs/multitask_ef_temporal_gradcam_evaluation" if Path("/kaggle/working").exists() else PROJECT_ROOT / "outputs" / "runs" / "multitask_ef_temporal_gradcam_evaluation",
))
MANIFEST_DIR = RUN_DIR / "manifests"
FIGURE_DIR = RUN_DIR / "figures"
MASK_DIR = RUN_DIR / "lv_masks_npz"
for directory in [RUN_DIR, MANIFEST_DIR, FIGURE_DIR, MASK_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"Project root: {PROJECT_ROOT}")
print(f"Raw EchoNet directory: {RAW_DIR}")
print(f"Processed directory: {PROCESSED_DIR}")
print(f"Trained multitask run: {TRAINED_RUN_DIR}")
print(f"Segmentation-only run: {SEGMENTATION_RUN_DIR}")
print(f"Reference Grad-CAM run: {REFERENCE_GRADCAM_RUN_DIR}")
print(f"Output directory: {RUN_DIR}")


## Configuration


In [ ]:
RUN_MODE = "smoke"  # change to "full" for the official test set

CONFIG = {
    "seed": 42,
    "num_frames_before": 11,
    "num_frames_after": 11,
    "temporal_stride": 2,
    "target_idx": 11,
    "sequence_length": 23,
    "image_size": [112, 112],
    "channels": [16, 32, 64, 128],
    "ef_hidden_dim": 128,
    "dropout": 0.1,
    "motion_hidden_channels": 64,
    "segmentation_threshold": 0.5,
    "saliency_fixed_threshold": 0.5,
    "primary_metric_cam_key": "positive_cams",
    "threshold_metric_cam_key": "clip_normalized_cams",
    "batch_size_for_inference": 4,
    "dynamic_segmentation_batch_size": 4,
    "num_workers": 2,
    "smoke_max_samples": 6,
    "representative_sample_count": 10,
    "save_gradcam_npz": True,
    "save_lv_mask_npz": True,
    "overlay_alpha": 0.45,
}

run_config_path = TRAINED_RUN_DIR / "config.json"
if run_config_path.exists():
    trained_config = json.loads(run_config_path.read_text())
    for key in [
        "num_frames_before", "num_frames_after", "temporal_stride", "target_idx", "sequence_length",
        "image_size", "channels", "ef_hidden_dim", "dropout", "motion_hidden_channels",
        "segmentation_threshold",
    ]:
        if key in trained_config:
            CONFIG[key] = trained_config[key]

CONFIG["run_mode"] = RUN_MODE
set_seed(int(CONFIG["seed"]))
with (RUN_DIR / "config.json").open("w", encoding="utf-8") as f:
    json.dump(CONFIG, f, indent=2)
CONFIG


## Dataset Reconstruction

The dataset and preprocessing mirror notebook 14: official EchoNet-Dynamic test split, 23 frames centered on the labelled ED/ES frame, stride 2.


In [ ]:
metadata_path = PROCESSED_DIR / "metadata.csv"
assert metadata_path.exists(), f"Missing processed metadata: {metadata_path}"
assert (RAW_DIR / "FileList.csv").exists(), f"Missing FileList.csv under {RAW_DIR}"
assert VIDEOS_DIR.exists(), f"Missing Videos directory: {VIDEOS_DIR}"

samples = load_temporal_metadata(metadata_path)
file_list, _volume_tracings = load_echonet_tables(RAW_DIR)
assert "EF" in file_list.columns, "FileList.csv must contain EF labels."

echo_table = file_list.copy()
echo_table["video_stem"] = echo_table["FileName"].astype(str).map(lambda x: Path(x).stem)
ef_lookup = dict(zip(echo_table["video_stem"], echo_table["EF"].astype(float)))

samples_with_ef = []
for sample in samples:
    item = dict(sample)
    video_stem = Path(str(item["video_id"])).stem
    if video_stem in ef_lookup and pd.notna(ef_lookup[video_stem]):
        item["ef"] = float(ef_lookup[video_stem])
        samples_with_ef.append(item)

gt_mask_lookup = {}
for sample in samples_with_ef:
    gt_mask_lookup[(str(sample["video_id"]), int(sample["frame_idx"]))] = str(sample["mask"])

train_samples, val_samples, test_samples = split_by_echonet_filelist(samples_with_ef, file_list)
ef_values_train = np.array([float(sample["ef"]) for sample in train_samples], dtype=np.float32)
ef_mean = float(ef_values_train.mean())
ef_std = float(ef_values_train.std(ddof=0))
assert ef_std > 0, "Training EF standard deviation is zero."

base_test_dataset = EchoNetTemporalDataset(
    test_samples,
    videos_dir=VIDEOS_DIR,
    num_frames_before=int(CONFIG["num_frames_before"]),
    num_frames_after=int(CONFIG["num_frames_after"]),
    temporal_stride=int(CONFIG["temporal_stride"]),
    image_size=tuple(CONFIG["image_size"]),
    augment=False,
)
test_dataset = EchoNetTemporalEFDataset(base_test_dataset, ef_mean=ef_mean, ef_std=ef_std)
sample_id_to_index = {str(sample["id"]): idx for idx, sample in enumerate(test_dataset.base_dataset.samples)}

if RUN_MODE == "smoke":
    active_indices = list(range(min(int(CONFIG["smoke_max_samples"]), len(test_dataset))))
else:
    active_indices = list(range(len(test_dataset)))
active_dataset = Subset(test_dataset, active_indices)
prediction_loader = DataLoader(
    active_dataset,
    batch_size=int(CONFIG["batch_size_for_inference"]),
    shuffle=False,
    num_workers=int(CONFIG["num_workers"]),
    pin_memory=torch.cuda.is_available(),
)
print({
    "train": len(train_samples),
    "validation": len(val_samples),
    "test": len(test_samples),
    "active": len(active_dataset),
    "run_mode": RUN_MODE,
    "ef_mean": ef_mean,
    "ef_std": ef_std,
})


## Load Both Multitask Models and Segmentation-Only Model


In [ ]:
checkpoint_paths = {
    "ef_primary": Path(os.environ.get(
        "EF_PRIMARY_CHECKPOINT_PATH",
        TRAINED_RUN_DIR / "checkpoints" / "ef_primary" / "best_ef_mae.pt",
    )),
    "ef_primary_motion": Path(os.environ.get(
        "EF_PRIMARY_MOTION_CHECKPOINT_PATH",
        TRAINED_RUN_DIR / "checkpoints" / "ef_primary_motion" / "best_ef_mae.pt",
    )),
}
for name, path in checkpoint_paths.items():
    if not path.exists():
        candidates = sorted(TRAINED_RUN_DIR.rglob(f"*{name}*best*ef*.pt")) + sorted(TRAINED_RUN_DIR.rglob("best_ef_mae.pt"))
        assert candidates, f"Missing {name} checkpoint. Set the corresponding checkpoint environment variable."
        checkpoint_paths[name] = candidates[0]

model_specs = {
    "ef_primary": {"with_motion": False},
    "ef_primary_motion": {"with_motion": True},
}
models = {}
checkpoint_metadata = {}
for model_name, spec in model_specs.items():
    model = build_ef_regression_convlstm(CONFIG, with_motion=bool(spec["with_motion"])).to(device)
    metadata = load_exact_checkpoint(model, checkpoint_paths[model_name], device="cpu")
    model.to(device)
    model.eval()
    models[model_name] = model
    checkpoint_metadata[model_name] = metadata
    print(model_name, json.dumps({k: v for k, v in metadata.items() if k not in {"config", "metrics"}}, indent=2, default=str))

seg_config_path = SEGMENTATION_RUN_DIR / "config.json"
seg_config = json.loads(seg_config_path.read_text()) if seg_config_path.exists() else {}
seg_checkpoint_path = Path(os.environ.get(
    "BIDIRECTIONAL_CONVLSTM_SEG_CHECKPOINT_PATH",
    SEGMENTATION_RUN_DIR / "checkpoints" / "best_model.pt",
))
if not seg_checkpoint_path.exists():
    candidates = sorted((SEGMENTATION_RUN_DIR / "checkpoints").glob("*.pt")) if (SEGMENTATION_RUN_DIR / "checkpoints").exists() else []
    assert candidates, f"Missing segmentation checkpoint under {SEGMENTATION_RUN_DIR}."
    seg_checkpoint_path = candidates[0]
segmentation_model = build_bidirectional_convlstm_unet(
    in_channels=1,
    out_channels=1,
    channels=tuple(seg_config.get("channels", CONFIG["channels"])),
    num_frames_before=int(seg_config.get("num_frames_before", CONFIG["num_frames_before"])),
    num_frames_after=int(seg_config.get("num_frames_after", CONFIG["num_frames_after"])),
).to(device)
seg_checkpoint = torch.load(seg_checkpoint_path, map_location="cpu")
seg_state = seg_checkpoint.get("model_state_dict", seg_checkpoint.get("state_dict", seg_checkpoint))
if any(key.startswith("module.") for key in seg_state):
    seg_state = {key.removeprefix("module."): value for key, value in seg_state.items()}
segmentation_model.load_state_dict(seg_state, strict=True)
segmentation_model.eval()
assert segmentation_model.expected_sequence_length == int(CONFIG["sequence_length"]), segmentation_model.expected_sequence_length
segmentation_checkpoint_metadata = {
    "checkpoint_path": str(seg_checkpoint_path),
    "checkpoint_keys": sorted(list(seg_checkpoint.keys())) if isinstance(seg_checkpoint, dict) else [],
    "epoch": seg_checkpoint.get("epoch") if isinstance(seg_checkpoint, dict) else None,
    "metrics": seg_checkpoint.get("metrics") if isinstance(seg_checkpoint, dict) else None,
    "config": seg_config,
}
with (MANIFEST_DIR / "checkpoint_metadata.json").open("w", encoding="utf-8") as f:
    json.dump({"models": checkpoint_metadata, "segmentation_pseudo_labeler": segmentation_checkpoint_metadata}, f, indent=2, default=str)
print("segmentation_model", json.dumps({k: v for k, v in segmentation_checkpoint_metadata.items() if k not in {"config", "metrics"}}, indent=2, default=str))


## Normal Inference and Representative Samples

Representative samples are loaded from notebook 14's selected-sample manifest when available. If not available, the same selection strategy is reproduced from current test predictions.


In [ ]:
prediction_tables = {}
dataset_metrics = {}
for model_name, model in models.items():
    predictions, metrics = run_normal_inference(model, prediction_loader, device, ef_mean, ef_std, model_name=model_name)
    prediction_tables[model_name] = predictions
    dataset_metrics[model_name] = metrics
    predictions.to_csv(MANIFEST_DIR / f"{model_name}_normal_test_predictions.csv", index=False)
    print(model_name, json.dumps(metrics, indent=2))
with (MANIFEST_DIR / "normal_test_metrics.json").open("w", encoding="utf-8") as f:
    json.dump(dataset_metrics, f, indent=2)

reference_selected_path = REFERENCE_GRADCAM_RUN_DIR / "manifests" / "selected_gradcam_samples.csv"
selected_by_model = {}
selected_rows = []
if reference_selected_path.exists():
    reference_selected = pd.read_csv(reference_selected_path)
    for model_name in models:
        model_selected = reference_selected[reference_selected["model_name"] == model_name].sort_values("selection_rank")
        sample_ids = [sid for sid in model_selected["sample_id"].astype(str).tolist() if sid in sample_id_to_index]
        if not sample_ids:
            sample_ids = select_representative_samples(prediction_tables[model_name], count=int(CONFIG["representative_sample_count"]))
        selected_by_model[model_name] = sample_ids[: int(CONFIG["representative_sample_count"])]
else:
    for model_name in models:
        selected_by_model[model_name] = select_representative_samples(prediction_tables[model_name], count=int(CONFIG["representative_sample_count"]))

for model_name, sample_ids in selected_by_model.items():
    for rank, sample_id in enumerate(sample_ids):
        row = prediction_tables[model_name][prediction_tables[model_name]["sample_id"] == sample_id].iloc[0].to_dict()
        row["selection_rank"] = rank
        selected_rows.append(row)
selected_df = pd.DataFrame(selected_rows)
selected_df.to_csv(MANIFEST_DIR / "selected_temporal_eval_samples.csv", index=False)
display(selected_df)


## Sliding-Window LV Masks

For each sampled input frame, the segmentation-only ConvLSTM predicts a pseudo-label by shifting that frame into the center of a 23-frame sequence. If a ground-truth ED/ES mask exists for that video/frame, the ground-truth mask replaces the prediction.


In [ ]:
def read_gt_mask(video_id: str, frame_idx: int) -> np.ndarray | None:
    path = gt_mask_lookup.get((str(video_id), int(frame_idx)))
    if path is None:
        return None
    mask = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if mask is None:
        return None
    mask = cv2.resize(mask, (int(CONFIG["image_size"][1]), int(CONFIG["image_size"][0])), interpolation=cv2.INTER_NEAREST)
    return (mask > 0).astype(np.uint8)


@torch.inference_mode()
def generate_dynamic_frame_masks(video_id: str, sampled_frame_indices: np.ndarray) -> dict[str, np.ndarray]:
    contexts = []
    context_indices = []
    video_path = base_test_dataset._video_path(str(video_id))
    for frame_idx in sampled_frame_indices.astype(int).tolist():
        sequence, indices, _frame_count = base_test_dataset._read_sequence(video_path, int(frame_idx))
        contexts.append(sequence[:, None].astype(np.float32))
        context_indices.append(np.asarray(indices, dtype=np.int32))
    contexts_np = np.stack(contexts, axis=0)
    probs = []
    batch_size = int(CONFIG["dynamic_segmentation_batch_size"])
    for start in range(0, contexts_np.shape[0], batch_size):
        chunk = torch.from_numpy(contexts_np[start:start + batch_size]).to(device)
        logits = segmentation_model(chunk)
        probs.append(torch.sigmoid(logits).detach().cpu().numpy()[:, 0].astype(np.float32))
    pred_prob = np.concatenate(probs, axis=0)
    pred_mask = (pred_prob >= float(CONFIG["segmentation_threshold"])).astype(np.uint8)
    roi_masks = pred_mask.copy()
    has_gt = np.zeros(len(sampled_frame_indices), dtype=np.uint8)
    roi_source = np.array(["predicted"] * len(sampled_frame_indices), dtype="U16")
    for i, frame_idx in enumerate(sampled_frame_indices.astype(int).tolist()):
        gt = read_gt_mask(str(video_id), int(frame_idx))
        if gt is not None:
            roi_masks[i] = gt.astype(np.uint8)
            has_gt[i] = 1
            roi_source[i] = "ground_truth"
    return {
        "dynamic_context_frame_indices": np.stack(context_indices, axis=0).astype(np.int32),
        "dynamic_frame_seg_prob": pred_prob.astype(np.float32),
        "dynamic_frame_seg_mask_pred": pred_mask.astype(np.uint8),
        "dynamic_frame_roi_mask": roi_masks.astype(np.uint8),
        "dynamic_frame_has_ground_truth": has_gt,
        "dynamic_frame_roi_source": roi_source,
    }


mask_cache: dict[str, dict[str, np.ndarray]] = {}
def get_or_create_lv_masks(batch: dict[str, Any]) -> tuple[dict[str, np.ndarray], Path]:
    sample_id = str(batch["id"][0])
    if sample_id in mask_cache:
        return mask_cache[sample_id], MASK_DIR / f"{sample_id}_dynamic_lv_masks.npz"
    out_path = MASK_DIR / f"{sample_id}_dynamic_lv_masks.npz"
    video_id = str(batch["video_id"][0])
    frame_indices = batch["frame_indices"][0].detach().cpu().numpy().astype(np.int32)
    masks = generate_dynamic_frame_masks(video_id, frame_indices)
    masks["sampled_frame_indices"] = frame_indices
    masks["sample_id"] = np.array(sample_id)
    masks["video_id"] = np.array(video_id)
    masks["target_idx"] = np.array(int(batch["target_idx"][0]), dtype=np.int32)
    masks["target_frame_idx"] = np.array(int(batch["frame_idx"][0]), dtype=np.int32)
    if bool(CONFIG["save_lv_mask_npz"]):
        np.savez_compressed(out_path, **masks)
    mask_cache[sample_id] = masks
    return masks, out_path


## Temporal Metric Definitions

Primary quantitative metrics use raw positive Grad-CAMs (`positive_cams`). Metric-specific normalization is used only where needed: temporal saliency IoU thresholds the clip-level normalized positive CAMs (`clip_normalized_cams`) on a common `[0,1]` scale. Independently frame-normalized CAMs and signed CAMs are not used for the primary quantitative analysis.


In [ ]:
def fixed_threshold_masks(cams: np.ndarray, threshold: float) -> np.ndarray:
    cams = np.asarray(cams, dtype=np.float32)
    return cams >= float(threshold)


def adjacent_iou(binary_masks: np.ndarray, eps: float = 1e-7) -> np.ndarray:
    values = []
    for t in range(binary_masks.shape[0] - 1):
        a = binary_masks[t].astype(bool)
        b = binary_masks[t + 1].astype(bool)
        inter = np.logical_and(a, b).sum()
        union = np.logical_or(a, b).sum()
        values.append(float((inter + eps) / (union + eps)))
    return np.asarray(values, dtype=np.float64)


def weighted_centroids(raw_positive_cams: np.ndarray) -> np.ndarray:
    return np.stack([centroid_from_map(np.clip(cam, 0.0, None)) for cam in raw_positive_cams], axis=0)


def center_saliency_mask_overlap(raw_positive_cams: np.ndarray, lv_masks: np.ndarray, target_idx: int, eps: float = 1e-7) -> float:
    cam = np.asarray(raw_positive_cams[target_idx], dtype=np.float64)
    mask = lv_masks[target_idx].astype(bool)
    total = float(cam.sum())
    return float(cam[mask].sum() / (total + eps))


def per_frame_lv_saliency_overlap(raw_positive_cams: np.ndarray, lv_masks: np.ndarray, eps: float = 1e-7) -> np.ndarray:
    values = []
    for t in range(raw_positive_cams.shape[0]):
        cam = np.asarray(raw_positive_cams[t], dtype=np.float64)
        mask = lv_masks[t].astype(bool)
        total = float(cam.sum())
        values.append(float(cam[mask].sum() / (total + eps)))
    return np.asarray(values, dtype=np.float64)


def compute_temporal_metrics(
    raw_positive_cams: np.ndarray,
    clip_normalized_cams: np.ndarray,
    lv_masks: np.ndarray,
    metadata: dict[str, Any],
) -> tuple[dict[str, Any], pd.DataFrame, pd.DataFrame]:
    raw_positive_cams = np.asarray(raw_positive_cams, dtype=np.float32)
    clip_normalized_cams = np.asarray(clip_normalized_cams, dtype=np.float32)
    lv_masks = np.asarray(lv_masks).astype(bool)
    assert raw_positive_cams.ndim == 3 and raw_positive_cams.shape == lv_masks.shape, (raw_positive_cams.shape, lv_masks.shape)
    assert clip_normalized_cams.shape == raw_positive_cams.shape, (clip_normalized_cams.shape, raw_positive_cams.shape)
    target_idx = int(metadata["target_idx"])

    # Raw positive CAMs are the primary quantitative source.
    consistency = adjacent_pearson(raw_positive_cams)
    centroids = weighted_centroids(raw_positive_cams)
    centroid_steps = np.linalg.norm(np.diff(centroids, axis=0), axis=1)
    lv_overlap = per_frame_lv_saliency_overlap(raw_positive_cams, lv_masks)
    center_overlap = center_saliency_mask_overlap(raw_positive_cams, lv_masks, target_idx)

    # Use clip-level normalized CAMs only where a common [0,1] threshold is required.
    saliency_masks = fixed_threshold_masks(clip_normalized_cams, float(CONFIG["saliency_fixed_threshold"]))
    ious = adjacent_iou(saliency_masks)

    row = dict(metadata)
    row.update({
        "saliency_consistency_mean": float(np.nanmean(consistency)) if consistency.size else float("nan"),
        "saliency_consistency_std": float(np.nanstd(consistency)) if consistency.size else float("nan"),
        "saliency_centroid_motion_mean": float(np.nanmean(centroid_steps)) if centroid_steps.size else float("nan"),
        "saliency_centroid_motion_std": float(np.nanstd(centroid_steps)) if centroid_steps.size else float("nan"),
        "center_saliency_mask_overlap": float(center_overlap),
        "lv_saliency_overlap_mean": float(np.nanmean(lv_overlap)),
        "lv_saliency_overlap_std": float(np.nanstd(lv_overlap)),
        "temporal_saliency_iou_mean": float(np.nanmean(ious)) if ious.size else float("nan"),
        "temporal_saliency_iou_std": float(np.nanstd(ious)) if ious.size else float("nan"),
        "saliency_fixed_threshold": float(CONFIG["saliency_fixed_threshold"]),
        "primary_metric_cam_key": str(CONFIG["primary_metric_cam_key"]),
        "threshold_metric_cam_key": str(CONFIG["threshold_metric_cam_key"]),
        "signed_cam_usage": "supplementary_visualization_only",
        "frame_normalized_cam_usage": "visualization_diagnostic_only",
    })
    for t, (x, y) in enumerate(centroids):
        row[f"saliency_centroid_x_frame_{t}"] = float(x)
        row[f"saliency_centroid_y_frame_{t}"] = float(y)

    transition_df = pd.DataFrame({
        "transition_idx": np.arange(len(ious), dtype=int),
        "saliency_consistency": consistency,
        "temporal_saliency_iou": ious,
        "centroid_step": centroid_steps,
    })
    for key, value in metadata.items():
        transition_df[key] = value
    transition_df["primary_metric_cam_key"] = str(CONFIG["primary_metric_cam_key"])
    transition_df["threshold_metric_cam_key"] = str(CONFIG["threshold_metric_cam_key"])

    frame_df = pd.DataFrame({
        "timestep": np.arange(raw_positive_cams.shape[0], dtype=int),
        "centroid_x": centroids[:, 0],
        "centroid_y": centroids[:, 1],
        "lv_saliency_overlap": lv_overlap,
        "saliency_mask_area_fraction": saliency_masks.reshape(raw_positive_cams.shape[0], -1).mean(axis=1),
        "lv_mask_area_fraction": lv_masks.reshape(raw_positive_cams.shape[0], -1).mean(axis=1),
        "raw_positive_cam_mass": raw_positive_cams.reshape(raw_positive_cams.shape[0], -1).sum(axis=1),
        "raw_positive_cam_max": raw_positive_cams.reshape(raw_positive_cams.shape[0], -1).max(axis=1),
    })
    for key, value in metadata.items():
        frame_df[key] = value
    frame_df["primary_metric_cam_key"] = str(CONFIG["primary_metric_cam_key"])
    frame_df["threshold_metric_cam_key"] = str(CONFIG["threshold_metric_cam_key"])
    return row, transition_df, frame_df


def summarize_metric_table(per_sample_df: pd.DataFrame) -> pd.DataFrame:
    metric_cols = [
        "saliency_consistency_mean",
        "saliency_centroid_motion_mean",
        "center_saliency_mask_overlap",
        "lv_saliency_overlap_mean",
        "temporal_saliency_iou_mean",
    ]
    rows = []
    for (model_name, cam_type), group in per_sample_df.groupby(["model_name", "cam_type"], dropna=False):
        for metric in metric_cols:
            values = pd.to_numeric(group[metric], errors="coerce").dropna().to_numpy(dtype=float)
            n = len(values)
            mean = float(np.mean(values)) if n else float("nan")
            std = float(np.std(values, ddof=1)) if n > 1 else float("nan")
            se = std / math.sqrt(n) if n > 1 and np.isfinite(std) else float("nan")
            ci = 1.96 * se if np.isfinite(se) else float("nan")
            rows.append({
                "model_name": model_name,
                "cam_type": cam_type,
                "metric": metric,
                "n": int(n),
                "mean": mean,
                "std": std,
                "median": float(np.median(values)) if n else float("nan"),
                "ci95_low": mean - ci if np.isfinite(ci) else float("nan"),
                "ci95_high": mean + ci if np.isfinite(ci) else float("nan"),
            })
    return pd.DataFrame(rows)


## Generate Grad-CAMs and Temporal Metrics

This is the main evaluation loop. It computes both target layers for both models. The temporal-representation maps remain timestep EF probes, exactly as in notebook 14.


In [ ]:
cam_specs = [
    ("encoder_bottleneck", "bottleneck_encoder"),
    ("temporal_representation", "fused_bidirectional_temporal_representation"),
]

per_sample_rows = []
transition_tables = []
frame_tables = []
manifest_rows = []
all_diagnostic_rows = []

for model_name, model in models.items():
    model_root = RUN_DIR / model_name
    for active_idx in tqdm(active_indices, desc=f"temporal Grad-CAM {model_name}"):
        batch = next(iter(DataLoader(Subset(test_dataset, [active_idx]), batch_size=1, shuffle=False, num_workers=0)))
        sample_id = str(batch["id"][0])
        sequence = batch["sequence"].to(device)
        assert sequence.shape == (1, int(CONFIG["sequence_length"]), 1, *tuple(CONFIG["image_size"])), tuple(sequence.shape)
        mask_payload, mask_npz_path = get_or_create_lv_masks(batch)
        lv_masks = mask_payload["dynamic_frame_roi_mask"].astype(bool)
        frame_indices = batch["frame_indices"][0].detach().cpu().numpy().astype(np.int32)
        pred_row = prediction_tables[model_name][prediction_tables[model_name]["sample_id"] == sample_id].iloc[0]

        for cam_type, target_layer in cam_specs:
            if cam_type == "encoder_bottleneck":
                result = encoder_bottleneck_ef_gradcam(model, sequence, ef_mean=ef_mean, ef_std=ef_std)
            else:
                result = temporal_representation_ef_probe_gradcam(model, sequence, ef_mean=ef_mean, ef_std=ef_std)
            assert abs(float(result.pred_ef) - float(pred_row["ef_pred"])) < 1e-3, (model_name, sample_id, cam_type, result.pred_ef, pred_row["ef_pred"])

            cam_dir = model_root / cam_type
            npz_path = cam_dir / "npz" / f"{sample_id}_{cam_type}_gradcam.npz"
            diagnostics_csv = cam_dir / "diagnostics" / f"{sample_id}_{cam_type}_diagnostics.csv"
            diagnostics_csv.parent.mkdir(parents=True, exist_ok=True)
            result.temporal_diagnostics.to_csv(diagnostics_csv, index=False)
            if bool(CONFIG["save_gradcam_npz"]):
                row = save_gradcam_npz(
                    npz_path,
                    result,
                    batch,
                    model_name=model_name,
                    checkpoint_metadata=checkpoint_metadata[model_name],
                    cam_type=cam_type,
                    target_layer=target_layer,
                    dataset_metrics=dataset_metrics[model_name],
                    diagnostics_csv_path=diagnostics_csv,
                )
            else:
                row = {
                    "sample_id": sample_id,
                    "video_id": str(batch["video_id"][0]),
                    "model_name": model_name,
                    "cam_type": cam_type,
                    "target_layer": target_layer,
                    "npz_path": "",
                    "diagnostics_csv_path": str(diagnostics_csv),
                }

            raw_positive_cams = result.positive_cams
            clip_normalized_cams = result.clip_normalized_cams
            metadata = {
                "model_name": model_name,
                "cam_type": cam_type,
                "target_layer": target_layer,
                "sample_id": sample_id,
                "video_id": str(batch["video_id"][0]),
                "target_idx": int(batch["target_idx"][0]),
                "target_frame_idx": int(batch["frame_idx"][0]),
                "ef_true": float(batch["ef"][0]),
                "ef_pred": float(result.pred_ef),
                "abs_ef_error": float(abs(float(result.pred_ef) - float(batch["ef"][0]))),
                "dynamic_mask_npz_path": str(mask_npz_path) if bool(CONFIG["save_lv_mask_npz"]) else "",
                "gt_mask_frame_count": int(mask_payload["dynamic_frame_has_ground_truth"].sum()),
            }
            metric_row, transition_df, frame_df = compute_temporal_metrics(raw_positive_cams, clip_normalized_cams, lv_masks, metadata)
            per_sample_rows.append(metric_row)
            transition_tables.append(transition_df)
            frame_tables.append(frame_df)

            diagnostic_df = result.temporal_diagnostics.copy()
            diagnostic_df.insert(0, "model_name", model_name)
            diagnostic_df.insert(1, "cam_type", cam_type)
            diagnostic_df.insert(2, "sample_id", sample_id)
            all_diagnostic_rows.append(diagnostic_df)

            row.update({
                "primary_metric_cam_key": str(CONFIG["primary_metric_cam_key"]),
                "threshold_metric_cam_key": str(CONFIG["threshold_metric_cam_key"]),
                "saliency_fixed_threshold": float(CONFIG["saliency_fixed_threshold"]),
                "dynamic_mask_npz_path": str(mask_npz_path) if bool(CONFIG["save_lv_mask_npz"]) else "",
                "gt_mask_frame_count": int(mask_payload["dynamic_frame_has_ground_truth"].sum()),
                "sampled_frame_indices": json.dumps(frame_indices.tolist()),
            })
            manifest_rows.append(row)

per_sample_df = pd.DataFrame(per_sample_rows)
transition_metrics_df = pd.concat(transition_tables, ignore_index=True) if transition_tables else pd.DataFrame()
frame_metrics_df = pd.concat(frame_tables, ignore_index=True) if frame_tables else pd.DataFrame()
cam_manifest_df = pd.DataFrame(manifest_rows)
temporal_diagnostics_df = pd.concat(all_diagnostic_rows, ignore_index=True) if all_diagnostic_rows else pd.DataFrame()
summary_df = summarize_metric_table(per_sample_df)

per_sample_df.to_csv(MANIFEST_DIR / "temporal_gradcam_per_sample_metrics.csv", index=False)
transition_metrics_df.to_csv(MANIFEST_DIR / "temporal_gradcam_transition_metrics.csv", index=False)
frame_metrics_df.to_csv(MANIFEST_DIR / "temporal_gradcam_frame_metrics.csv", index=False)
cam_manifest_df.to_csv(MANIFEST_DIR / "temporal_gradcam_manifest.csv", index=False)
temporal_diagnostics_df.to_csv(MANIFEST_DIR / "temporal_gradcam_diagnostics.csv", index=False)
summary_df.to_csv(MANIFEST_DIR / "temporal_gradcam_dataset_summary.csv", index=False)
display(summary_df)
display(per_sample_df.head())


## Paired Statistical Comparisons


In [ ]:
def paired_test(values_a: np.ndarray, values_b: np.ndarray) -> dict[str, Any]:
    a = np.asarray(values_a, dtype=float)
    b = np.asarray(values_b, dtype=float)
    mask = np.isfinite(a) & np.isfinite(b)
    a = a[mask]
    b = b[mask]
    diff = b - a
    n = len(diff)
    if n < 2:
        return {"n": int(n), "test": "insufficient_pairs", "statistic": float("nan"), "p_value": float("nan"), "effect_size": float("nan"), "mean_difference_b_minus_a": float("nan")}
    normal = False
    if scipy_stats is not None and 3 <= n <= 5000:
        try:
            normal = bool(scipy_stats.shapiro(diff).pvalue > 0.05)
        except Exception:
            normal = False
    if scipy_stats is not None and normal:
        test = scipy_stats.ttest_rel(b, a, nan_policy="omit")
        statistic = float(test.statistic)
        p_value = float(test.pvalue)
        test_name = "paired_t_test"
    elif scipy_stats is not None:
        try:
            test = scipy_stats.wilcoxon(b, a, zero_method="wilcox", alternative="two-sided")
            statistic = float(test.statistic)
            p_value = float(test.pvalue)
            test_name = "wilcoxon_signed_rank"
        except Exception:
            test = scipy_stats.ttest_rel(b, a, nan_policy="omit")
            statistic = float(test.statistic)
            p_value = float(test.pvalue)
            test_name = "paired_t_test_fallback"
    else:
        statistic = float("nan")
        p_value = float("nan")
        test_name = "scipy_unavailable"
    effect = float(np.nanmean(diff) / np.nanstd(diff, ddof=1)) if n > 1 and np.nanstd(diff, ddof=1) > 1e-12 else float("nan")
    return {
        "n": int(n),
        "test": test_name,
        "statistic": statistic,
        "p_value": p_value,
        "effect_size_cohens_dz": effect,
        "mean_difference_b_minus_a": float(np.nanmean(diff)),
        "median_difference_b_minus_a": float(np.nanmedian(diff)),
    }

metric_cols = [
    "saliency_consistency_mean",
    "saliency_centroid_motion_mean",
    "center_saliency_mask_overlap",
    "lv_saliency_overlap_mean",
    "temporal_saliency_iou_mean",
]
comparison_rows = []
# Model comparison within each CAM type.
for cam_type in per_sample_df["cam_type"].dropna().unique():
    wide = per_sample_df[per_sample_df["cam_type"] == cam_type].pivot_table(index="sample_id", columns="model_name", values=metric_cols, aggfunc="first")
    for metric in metric_cols:
        if (metric, "ef_primary") in wide.columns and (metric, "ef_primary_motion") in wide.columns:
            result = paired_test(wide[(metric, "ef_primary")].to_numpy(), wide[(metric, "ef_primary_motion")].to_numpy())
            result.update({"comparison": "ef_primary_vs_ef_primary_motion", "cam_type": cam_type, "metric": metric, "a": "ef_primary", "b": "ef_primary_motion"})
            comparison_rows.append(result)

# Layer comparison within each model.
for model_name in per_sample_df["model_name"].dropna().unique():
    wide = per_sample_df[per_sample_df["model_name"] == model_name].pivot_table(index="sample_id", columns="cam_type", values=metric_cols, aggfunc="first")
    for metric in metric_cols:
        if (metric, "encoder_bottleneck") in wide.columns and (metric, "temporal_representation") in wide.columns:
            result = paired_test(wide[(metric, "encoder_bottleneck")].to_numpy(), wide[(metric, "temporal_representation")].to_numpy())
            result.update({"comparison": "encoder_bottleneck_vs_temporal_representation", "model_name": model_name, "metric": metric, "a": "encoder_bottleneck", "b": "temporal_representation"})
            comparison_rows.append(result)

stats_df = pd.DataFrame(comparison_rows)
stats_df.to_csv(MANIFEST_DIR / "temporal_gradcam_paired_statistics.csv", index=False)
display(stats_df)


## Representative Visualizations

Figures are generated only for the representative samples to keep the output size manageable.


In [ ]:
visual_rows = []
for model_name, sample_ids in selected_by_model.items():
    for sample_id in tqdm(sample_ids, desc=f"visualizations {model_name}"):
        for cam_type, _target_layer in cam_specs:
            row_match = cam_manifest_df[(cam_manifest_df["model_name"] == model_name) & (cam_manifest_df["sample_id"] == sample_id) & (cam_manifest_df["cam_type"] == cam_type)]
            if row_match.empty:
                continue
            npz_path = Path(str(row_match.iloc[0]["npz_path"]))
            if not npz_path.exists():
                continue
            cam_dir = RUN_DIR / model_name / cam_type
            overlay_path = cam_dir / "temporal_eval_overlays" / f"{sample_id}_{cam_type}_clip_normalized_overlay.png"
            make_gradcam_overlay_figure(
                npz_path,
                overlay_path,
                dataset_metrics[model_name],
                cam_key="clip_normalized_cams",
                overlay_name="positive clip-normalized temporal metric CAM",
                signed=False,
                positive_display_mode="enhanced",
                cmap_name="turbo",
                alpha=float(CONFIG["overlay_alpha"]),
            )
            diagnostic_plot_path = cam_dir / "temporal_eval_diagnostic_plots" / f"{sample_id}_{cam_type}_temporal_diagnostics.png"
            diagnostics_csv_path = Path(str(row_match.iloc[0]["diagnostics_csv_path"]))
            target_idx = int(row_match.iloc[0].get("target_idx", CONFIG["target_idx"])) if "target_idx" in row_match.columns else int(CONFIG["target_idx"])
            make_temporal_diagnostic_plot(
                diagnostics_csv_path,
                diagnostic_plot_path,
                target_idx=target_idx,
                title=f"{model_name} | {cam_type} | {sample_id}",
            )
            motion_trace_path = cam_dir / "temporal_eval_motion_trace" / f"{sample_id}_{cam_type}_motion_trace.png"
            centroid_df = make_motion_trace_overlay_figure(
                npz_path,
                motion_trace_path,
                dataset_metrics[model_name],
                cam_key="clip_normalized_cams",
                overlay_name="clip-normalized temporal metric CAM trace",
                alpha=float(CONFIG["overlay_alpha"]),
                positive_display_mode="enhanced",
            )
            centroid_csv = cam_dir / "temporal_eval_centroids" / f"{sample_id}_{cam_type}_centroids.csv"
            centroid_csv.parent.mkdir(parents=True, exist_ok=True)
            centroid_df.insert(0, "sample_id", sample_id)
            centroid_df.insert(1, "model_name", model_name)
            centroid_df.insert(2, "cam_type", cam_type)
            centroid_df.to_csv(centroid_csv, index=False)
            centroid_plot_path = cam_dir / "temporal_eval_centroid_plots" / f"{sample_id}_{cam_type}_centroid_trajectory.png"
            make_centroid_trajectory_plot(centroid_csv, centroid_plot_path, target_idx=target_idx, title=f"{model_name} | {cam_type} | {sample_id}")

            sample_metrics = transition_metrics_df[(transition_metrics_df["model_name"] == model_name) & (transition_metrics_df["sample_id"] == sample_id) & (transition_metrics_df["cam_type"] == cam_type)]
            metric_plot_path = cam_dir / "temporal_eval_metric_plots" / f"{sample_id}_{cam_type}_temporal_metrics.png"
            metric_plot_path.parent.mkdir(parents=True, exist_ok=True)
            fig, ax1 = plt.subplots(figsize=(9, 4))
            ax1.plot(sample_metrics["transition_idx"], sample_metrics["saliency_consistency"], label="adjacent consistency", marker="o")
            ax1.plot(sample_metrics["transition_idx"], sample_metrics["temporal_saliency_iou"], label="temporal IoU", marker="o")
            ax1.set_xlabel("transition")
            ax1.set_ylabel("consistency / IoU")
            ax2 = ax1.twinx()
            ax2.plot(sample_metrics["transition_idx"], sample_metrics["centroid_step"], color="tab:red", label="centroid step", marker=".")
            ax2.set_ylabel("centroid motion (pixels)")
            lines1, labels1 = ax1.get_legend_handles_labels()
            lines2, labels2 = ax2.get_legend_handles_labels()
            ax1.legend(lines1 + lines2, labels1 + labels2, fontsize=8, loc="best")
            fig.suptitle(f"{model_name} | {cam_type} | {sample_id}")
            fig.tight_layout()
            fig.savefig(metric_plot_path, dpi=150, bbox_inches="tight")
            plt.close(fig)

            visual_rows.append({
                "model_name": model_name,
                "sample_id": sample_id,
                "cam_type": cam_type,
                "overlay_path": str(overlay_path),
                "diagnostic_plot_path": str(diagnostic_plot_path),
                "motion_trace_path": str(motion_trace_path),
                "centroid_csv_path": str(centroid_csv),
                "centroid_plot_path": str(centroid_plot_path),
                "metric_plot_path": str(metric_plot_path),
            })
visual_df = pd.DataFrame(visual_rows)
visual_df.to_csv(MANIFEST_DIR / "temporal_gradcam_visualization_manifest.csv", index=False)
display(visual_df.head())


## Final Output Check


In [ ]:
expected_outputs = [
    RUN_DIR / "config.json",
    MANIFEST_DIR / "checkpoint_metadata.json",
    MANIFEST_DIR / "normal_test_metrics.json",
    MANIFEST_DIR / "selected_temporal_eval_samples.csv",
    MANIFEST_DIR / "temporal_gradcam_manifest.csv",
    MANIFEST_DIR / "temporal_gradcam_per_sample_metrics.csv",
    MANIFEST_DIR / "temporal_gradcam_transition_metrics.csv",
    MANIFEST_DIR / "temporal_gradcam_frame_metrics.csv",
    MANIFEST_DIR / "temporal_gradcam_dataset_summary.csv",
    MANIFEST_DIR / "temporal_gradcam_paired_statistics.csv",
    MANIFEST_DIR / "temporal_gradcam_visualization_manifest.csv",
]
for path in expected_outputs:
    print(path, "exists=", path.exists())
summary = {
    "run_mode": RUN_MODE,
    "active_samples": int(len(active_dataset)),
    "models": list(models.keys()),
    "cam_types": [spec[0] for spec in cam_specs],
    "primary_metric_cam_key": str(CONFIG["primary_metric_cam_key"]),
    "threshold_metric_cam_key": str(CONFIG["threshold_metric_cam_key"]),
    "saliency_fixed_threshold": float(CONFIG["saliency_fixed_threshold"]),
    "quantitative_cam_note": "Raw positive_cams are used for saliency consistency, mass-weighted centroid motion, and LV saliency overlap. clip_normalized_cams are used only for fixed-threshold temporal saliency IoU. signed/frame-normalized CAMs are supplementary only.",
    "temporal_representation_note": "Temporal representation maps are timestep EF-head probes reused exactly from notebook 14; non-target timesteps are counterfactual probes.",
    "lv_mask_note": "Sliding-window segmentation-only ConvLSTM pseudo-labels are used for non-GT frames; GT ED/ES masks replace predictions whenever available.",
    "output_dir": str(RUN_DIR),
}
with (MANIFEST_DIR / "temporal_gradcam_evaluation_summary.json").open("w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))
